# SonicVitals rPPG benchmark (Google Colab)

Compares the **SonicVitals** camera pipeline with signal-processing baselines (GREEN, CHROM, POS; ported line for line from rPPG-Toolbox and verified identical) and pretrained **deep models** (TS-CAN, PhysNet, EfficientPhys from rPPG-Toolbox) on public datasets with contact-PPG ground truth.

Metrics: heart-rate **MAE / RMSE / MAPE / Pearson r**, share of windows within 5 BPM, **coverage** (share of windows where a method gives a value), **latency**, and **cross-dataset generalization** (deep models are only tested on datasets they were not trained on).

Everything runs on Google's machines: nothing is downloaded to your laptop.

**Before you start**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. Upload `SonicVitals_benchmark.zip` (from the project's `benchmark/` folder) to a folder in your Google Drive; results are saved next to it.
3. Open the [UBFC-rPPG Drive folder](https://drive.google.com/drive/folders/1o0XU4gTIo46YfwaWjIgbtCncc-oF44Xk), then *right-click the folder → Organize → Add shortcut → My Drive*. A shortcut uses **none** of your storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Code: SonicVitals + rPPG-Toolbox

In [ ]:
import os, glob, subprocess
# Your Drive layout (falls back to searching the whole Drive if the file is moved).
ZIP = '/content/drive/MyDrive/SonicVitals/benchmark/SonicVitals_benchmark.zip'
if not os.path.isfile(ZIP):
    hits = sorted(glob.glob('/content/drive/MyDrive/**/SonicVitals_benchmark.zip', recursive=True))
    assert hits, 'SonicVitals_benchmark.zip not found in your Drive'
    ZIP = hits[0]
WORK = os.path.dirname(ZIP)          # results and caches are written next to the zip
print('code:', ZIP)
!rm -rf /content/sv && mkdir -p /content/sv && unzip -q -o "$ZIP" -d /content/sv
# rPPG-Toolbox: pretrained weights + model code. Clone it; if that fails, use the copy uploaded to Drive.
TB = '/content/sv/benchmark/third_party/rPPG-Toolbox'
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ubicomplab/rPPG-Toolbox.git', TB],
                   capture_output=True, text=True)
if not os.path.isfile(TB + '/final_model_release/PURE_TSCAN.pth'):
    print('git clone failed:', r.stderr[-500:])
    drive_copy = [WORK + '/third_party/rPPG-Toolbox/final_model_release/PURE_TSCAN.pth']
    if not os.path.isfile(drive_copy[0]):
        drive_copy = sorted(glob.glob('/content/drive/MyDrive/**/rPPG-Toolbox/final_model_release/PURE_TSCAN.pth', recursive=True))
    assert drive_copy, 'rPPG-Toolbox not available: re-run this cell, or upload benchmark/third_party/rPPG-Toolbox to Drive'
    TB = drive_copy[0].rsplit('/final_model_release', 1)[0]
os.environ['RPPG_TOOLBOX'] = TB
print('rPPG-Toolbox:', TB, '|', len(glob.glob(TB + '/final_model_release/*.pth')), 'pretrained weights')
%cd /content/sv
!python -c "import torch, cv2, scipy, pandas; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| opencv', cv2.__version__)"

OUT = WORK + '/results'
CACHE = WORK + '/cache'

def run_bench(args, log_name):
    """Runs the benchmark, prints progress in this cell, saves the full log next to the results,
    and shows the error if the run fails."""
    import shlex
    log = f'{WORK}/{log_name}'
    proc = subprocess.Popen(['python', '-u', 'benchmark/run_benchmark.py'] + shlex.split(args), cwd='/content/sv',
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    with open(log, 'w') as f:
        for line in proc.stdout:
            f.write(line)
            lines.append(line)
            low = line.lower()
            if 'warn' not in low and 'no face' not in low:
                print(line, end='')
    rc = proc.wait()
    if rc != 0:
        print(f'\n*** benchmark failed (exit {rc}). Last lines of {log}: ***')
        print(''.join(lines[-40:]))
    return rc == 0


## 2. Quick self-test (2 minutes)
Synthetic videos in the UBFC layout with a known pulse. Every method should be within ~1 BPM; this only checks the code, not accuracy on real faces.

In [ ]:
!python benchmark/make_synthetic.py --out /content/synth --subjects 2 --seconds 35
run_bench('--dataset ubfc --root /content/synth --cache /content/cache_synth --out /content/results_synth '
          '--dl-train PURE --device cuda', 'selftest_log.txt')


## 3. UBFC-rPPG (42 subjects, 640×480 @ 30 fps, CMS50E finger-oximeter reference)
Finds the dataset through your Drive shortcut. Results and caches are written to the same Drive folder as the zip, so a disconnected session resumes where it stopped.

Deep models: weights trained on **PURE** and on **SCAMPS** (synthetic), both *cross-dataset* for UBFC.

In [ ]:
def find_ubfc():
    pats = ['/content/drive/MyDrive/**/subject*/vid.avi', '/content/drive/.shortcut-targets-by-id/**/subject*/vid.avi']
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(sorted(hits)[0]))
    return None
UBFC = find_ubfc()
print('UBFC-rPPG root:', UBFC)
assert UBFC, 'Add the UBFC-rPPG folder as a shortcut in My Drive (see the top of this notebook)'
print(len(glob.glob(UBFC + '/subject*/vid.avi')), 'subjects found')

In [ ]:
# LIMIT = 10 subjects takes ~30-45 min on Colab; set LIMIT = None for all 42 (~2-3 h).
# Interrupted runs resume: finished videos are cached in Drive.
LIMIT = 10
lim = f' --limit {LIMIT}' if LIMIT else ''
ok = run_bench(f'--dataset ubfc --root "{UBFC}" --cache "{CACHE}" --out "{OUT}" '
               f'--dl-train PURE,SCAMPS --device cuda --window 30 --step 10 --sv-box-baselines{lim}', 'ubfc_log.txt')
if ok:
    !python benchmark/plots.py "$OUT/UBFC-rPPG"


In [ ]:
from IPython.display import Markdown, Image, display
if os.path.isfile(f'{OUT}/UBFC-rPPG/summary.md'):
    display(Markdown(open(f'{OUT}/UBFC-rPPG/summary.md').read()))
    display(Image(f'{OUT}/UBFC-rPPG/bland_altman.png'))
else:
    print('No UBFC results yet: see the error printed by the cell above.')


## 4. PURE (10 subjects × 6 motion tasks, 640×480 @ 30 fps, 60 Hz pulse oximeter)
PURE is released **on request** by TU Ilmenau (Neuroinformatics and Cognitive Robotics Lab). Once you have it, put the unpacked folders (`01-01`, `01-02`, …) in *My Drive/SonicVitals/PURE/*.

Deep models trained on **UBFC-rPPG** and **SCAMPS** (cross-dataset for PURE).

In [ ]:
PURE = WORK + '/PURE'
if os.path.isdir(PURE):
    if run_bench(f'--dataset pure --root "{PURE}" --cache "{CACHE}" --out "{OUT}" '
                 '--dl-train UBFC-rPPG,SCAMPS --device cuda --window 30 --step 10', 'pure_log.txt'):
        !python benchmark/plots.py "$OUT/PURE"
        display(Markdown(open(f'{OUT}/PURE/summary.md').read()))
else:
    print('PURE not available yet (request access, then upload to', PURE, ')')


## 5. VIPL-HR (107 subjects, 3 RGB cameras, head movement and lighting changes)
VIPL-HR is released under a **license agreement** that must be signed (normally by your supervisor) and e-mailed to the VIPL lab (ICT, Chinese Academy of Sciences). Put the `data/` folder (`p1/v1/source1/...`) in *My Drive/SonicVitals/VIPL-HR/*. The NIR camera (source4) is excluded.

In [ ]:
VIPL = WORK + '/VIPL-HR'
if os.path.isdir(VIPL):
    if run_bench(f'--dataset vipl --root "{VIPL}" --cache "{CACHE}" --out "{OUT}" '
                 '--dl-train PURE,UBFC-rPPG,SCAMPS --device cuda --window 30 --step 10', 'vipl_log.txt'):
        !python benchmark/plots.py "$OUT/VIPL-HR"
        display(Markdown(open(f'{OUT}/VIPL-HR/summary.md').read()))
else:
    print('VIPL-HR not available yet (license required), expected at', VIPL)


## 6. Cross-dataset generalization table (window level MAE in BPM, coverage in brackets)

In [ ]:
import pandas as pd
rows = []
for ds in ['UBFC-rPPG', 'PURE', 'VIPL-HR']:
    f = f'{OUT}/{ds}/summary.csv'
    if os.path.isfile(f):
        s = pd.read_csv(f); s = s[s.level == 'window']
        for _, r in s.iterrows():
            m = r['method'].split(' [')[0] + (' [' + r['method'].split(' [')[1].split('->')[0] + '-trained]' if ' [' in r['method'] else '')
            rows.append(dict(method=m, dataset=ds, cell=f"{r['MAE']:.2f} ({r['coverage']:.0%})"))
if rows:
    tab = pd.DataFrame(rows).pivot_table(index='method', columns='dataset', values='cell', aggfunc='first')
    display(tab)
    tab.to_csv(f'{OUT}/cross_dataset.csv')